Business Understanding
        ↓
Data Collection
        ↓
Dataset Understanding
        ↓
Data Cleaning
        ↓
Handle Missing Values
        ↓
Handle Outliers (if required)
        ↓
Exploratory Data Analysis (EDA)
        ↓
Feature Engineering (if required)
        ↓
Define Features (X) and Target (y)
        ↓
Train-Validation-Test Split
        ↓
Encoding
        ↓
Feature Scaling
        ↓
Convert to Tensors
        ↓
Create Dataset & DataLoader
        ↓
Design Neural Network Architecture
        ↓
Train the Model
        ↓
Evaluate the Model
        ↓
Save Model & Artifacts

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Using device:", device)

# Data Collection

In [ ]:
df = pd.read_csv("../data/medical_insurance_cost.csv")

pd.set_option("display.max_columns", None)

df.sample(3)

# Dataset Understanding

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
df.isnull().sum()

<h2 style="font-size:30px; color:#1E3A8A; border-bottom:2px solid #DBEAFE; padding-bottom:7px;">
    <strong>Dataset Column Descriptions</strong>
</h2>

<h3 style="color:#1D4ED8;">Demographic & Personal Information</h3>

<div style="background:linear-gradient(135deg,#EFF6FF,#F5F3FF); border-left:5px solid #2563EB; padding:14px 18px; border-radius:8px; line-height:1.9;">

<strong>person_id:</strong> Unique identifier assigned to each person in the dataset. Not used as a predictive feature.<br><br>
<strong>age:</strong> Age of the person in years.<br><br>
<strong>sex:</strong> Sex category of the person.<br><br>
<strong>region:</strong> Geographic region where the person is located.<br><br>
<strong>urban_rural:</strong> Classification of the person's residential area.<br><br>
<strong>income:</strong> Approximate annual income of the person.<br><br>
<strong>education:</strong> Highest education level attained by the person.<br><br>
<strong>marital_status:</strong> Marital status of the person.<br><br>
<strong>employment_status:</strong> Employment category of the person.<br><br>
<strong>household_size:</strong> Total number of people living in the person's household.<br><br>
<strong>dependents:</strong> Number of dependents associated with the person.

</div>

<h3 style="color:#15803D;">Health & Lifestyle Information</h3>

<div style="background:linear-gradient(135deg,#F0FDF4,#ECFDF5); border-left:5px solid #16A34A; padding:14px 18px; border-radius:8px; line-height:1.9;">

<strong>bmi:</strong> Body Mass Index of the person.<br><br>
<strong>smoker:</strong> Whether the person is a smoker.<br><br>
<strong>alcohol_freq:</strong> Frequency of alcohol consumption. Has a lot of missing values, dropped later.<br><br>
<strong>visits_last_year:</strong> Number of healthcare visits in the previous year.<br><br>
<strong>hospitalizations_last_3yrs:</strong> Number of hospitalizations in the previous three years.<br><br>
<strong>days_hospitalized_last_3yrs:</strong> Total days hospitalized in the previous three years.<br><br>
<strong>medication_count:</strong> Number of medications currently associated with the person.

</div>

<h3 style="color:#991B1B;">Clinical Measurements</h3>

<div style="background:linear-gradient(135deg,#FEF2F2,#FFF1F2); border-left:5px solid #DC2626; padding:14px 18px; border-radius:8px; line-height:1.9;">

<strong>systolic_bp:</strong> Systolic blood pressure.<br><br>
<strong>diastolic_bp:</strong> Diastolic blood pressure.<br><br>
<strong>ldl:</strong> LDL cholesterol level.<br><br>
<strong>hba1c:</strong> Average blood glucose level over 2-3 months.

</div>

<h3 style="color:#7E22CE;">Insurance Policy Information</h3>

<div style="background:linear-gradient(135deg,#FAF5FF,#F5F3FF); border-left:5px solid #9333EA; padding:14px 18px; border-radius:8px; line-height:1.9;">

<strong>plan_type:</strong> Type of insurance plan.<br><br>
<strong>network_tier:</strong> Tier of the healthcare provider network.<br><br>
<strong>deductible:</strong> Amount paid before coverage begins.<br><br>
<strong>copay:</strong> Fixed amount paid for a covered service.<br><br>
<strong>policy_term_years:</strong> Duration of the policy in years.<br><br>
<strong>policy_changes_last_2yrs:</strong> Number of policy changes in the previous two years.<br><br>
<strong>provider_quality:</strong> Score representing quality of the healthcare provider/network.<br><br>
<strong>risk_score:</strong> Estimated healthcare risk score. Dropped later, this leaks target information.

</div>

<h3 style="color:#A16207;">Medical Cost & Insurance Financial Information</h3>

<div style="background:linear-gradient(135deg,#FEFCE8,#FFFBEB); border-left:5px solid #EAB308; padding:14px 18px; border-radius:8px; line-height:1.9;">

<strong>annual_medical_cost:</strong> Total medical/healthcare expenditure over a year. <strong>This is the target variable.</strong><br><br>
<strong>annual_premium, monthly_premium, claims_count, avg_claim_amount, total_claims_paid:</strong> All dropped later since they are directly derived from the same billing process as the target and would leak information.

</div>

<h3 style="color:#0F766E;">Chronic Medical Conditions</h3>

<div style="background:linear-gradient(135deg,#F0FDFA,#ECFEFF); border-left:5px solid #0D9488; padding:14px 18px; border-radius:8px; line-height:1.9;">

<strong>chronic_count:</strong> Total number of chronic medical conditions.<br><br>
<strong>hypertension, diabetes, asthma, copd, cardiovascular_disease, cancer_history, kidney_disease, liver_disease, arthritis, mental_health:</strong> Binary indicators (1 = present, 0 = absent).

</div>

<h3 style="color:#C2410C;">Medical Procedure Information</h3>

<div style="background:linear-gradient(135deg,#FFF7ED,#FFEDD5); border-left:5px solid #F97316; padding:14px 18px; border-radius:8px; line-height:1.9;">

<strong>proc_imaging_count, proc_surgery_count, proc_physio_count, proc_consult_count, proc_lab_count:</strong> Counts of different medical procedures.<br><br>
<strong>is_high_risk:</strong> Dropped later, derived from risk_score.<br><br>
<strong>had_major_procedure:</strong> Dropped later, directly tied to the procedure counts and cost.

</div>

# Data Cleaning

In [ ]:
# alcohol_freq has a lot of missing values, dropping the column instead of imputing

print("alcohol_freq missing %:", df["alcohol_freq"].isnull().mean() * 100)

df.drop(columns=["alcohol_freq"], inplace=True)

In [ ]:
# columns that should not be used as input features
# person_id is just an identifier
# annual_premium, monthly_premium, claims_count, avg_claim_amount, total_claims_paid, risk_score, is_high_risk, had_major_procedure
# all of these are directly derived from the same billing/claims process as the target, keeping them would leak the answer

unwanted_columns = [
    "person_id",
    "annual_premium",
    "monthly_premium",
    "claims_count",
    "avg_claim_amount",
    "total_claims_paid",
    "risk_score",
    "is_high_risk",
    "had_major_procedure",
]

df.drop(columns=unwanted_columns, inplace=True)

df.shape

# Exploratory Data Analysis (EDA)

In [ ]:
sns.set_style("whitegrid")

plt.figure(figsize=(9, 5))
sns.histplot(df["annual_medical_cost"], kde=True, color="#2563EB", bins=50)
plt.title("Distribution of Annual Medical Cost", fontsize=14)
plt.xlabel("Annual Medical Cost")
plt.ylabel("Count")
plt.tight_layout()
plt.savefig("../images/01_target_distribution.png")
plt.show()

In [ ]:
numerical_columns = df.select_dtypes(include=["number"]).columns.tolist()
numerical_columns.remove("annual_medical_cost")

fig, axes = plt.subplots(7, 5, figsize=(24, 26))
axes = axes.flatten()

colors = sns.color_palette("husl", len(numerical_columns))

for i, col in enumerate(numerical_columns):
    sns.histplot(df[col], kde=True, ax=axes[i], color=colors[i])
    axes[i].set_title(col, fontsize=10)

for j in range(len(numerical_columns), len(axes)):
    fig.delaxes(axes[j])

plt.tight_layout()
plt.savefig("../images/02_numerical_distributions.png")
plt.show()

In [ ]:
categorical_columns = df.select_dtypes(include="object").columns.tolist()

fig, axes = plt.subplots(3, 3, figsize=(20, 16))
axes = axes.flatten()

for i, col in enumerate(categorical_columns):
    counts = df[col].value_counts()
    axes[i].pie(
        counts.values,
        labels=counts.index,
        autopct="%1.1f%%",
        colors=sns.color_palette("Set2", len(counts)),
        startangle=90,
    )
    axes[i].set_title(col, fontsize=12)

plt.tight_layout()
plt.savefig("../images/03_categorical_distributions.png")
plt.show()

In [ ]:
plt.figure(figsize=(18, 14))
corr = df[numerical_columns + ["annual_medical_cost"]].corr()
sns.heatmap(corr, cmap="coolwarm", center=0, annot=False, linewidths=0.3)
plt.title("Correlation Heatmap", fontsize=16)
plt.tight_layout()
plt.savefig("../images/04_correlation_heatmap.png")
plt.show()

In [ ]:
top_corr_features = corr["annual_medical_cost"].abs().sort_values(ascending=False)[1:7].index.tolist()

fig, axes = plt.subplots(2, 3, figsize=(20, 11))
axes = axes.flatten()

for i, col in enumerate(top_corr_features):
    sns.scatterplot(x=df[col], y=df["annual_medical_cost"], ax=axes[i], color="#7C3AED", alpha=0.4)
    axes[i].set_title(f"{col} vs annual_medical_cost", fontsize=11)

plt.tight_layout()
plt.savefig("../images/05_feature_vs_target.png")
plt.show()

# Define Features (X) and Target (y)

In [ ]:
X = df.drop(columns=["annual_medical_cost"])
y = df["annual_medical_cost"]

print("X shape:", X.shape)
print("y shape:", y.shape)

# Train-Validation-Test Split

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.3, random_state=42)

X_valid, X_test, y_valid, y_test = train_test_split(X_temp, y_temp, test_size=0.50, random_state=42)

In [ ]:
print("Training Set   :", X_train.shape)
print("Validation Set :", X_valid.shape)
print("Testing Set    :", X_test.shape)

print()

print("Training Labels   :", y_train.shape)
print("Validation Labels :", y_valid.shape)
print("Testing Labels    :", y_test.shape)

# Encoding

In [ ]:
categorical_columns = X_train.select_dtypes(include="object").columns.tolist()

print(categorical_columns)
print("\n==========================\n")

numerical_columns = X_train.select_dtypes(include=["number"]).columns.tolist()

numerical_columns

In [ ]:
from sklearn.preprocessing import OneHotEncoder

encoder = OneHotEncoder(
    handle_unknown="ignore",
    sparse_output=False
)

X_train_cat = encoder.fit_transform(X_train[categorical_columns])
X_valid_cat = encoder.transform(X_valid[categorical_columns])
X_test_cat = encoder.transform(X_test[categorical_columns])

X_train_cat.shape

# Feature Scaling

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_num = scaler.fit_transform(X_train[numerical_columns])
X_valid_num = scaler.transform(X_valid[numerical_columns])
X_test_num = scaler.transform(X_test[numerical_columns])

X_train_num.shape

In [ ]:
X_train_final = np.hstack([X_train_num, X_train_cat])
X_valid_final = np.hstack([X_valid_num, X_valid_cat])
X_test_final = np.hstack([X_test_num, X_test_cat])

encoded_cat_names = encoder.get_feature_names_out(categorical_columns).tolist()
feature_order = numerical_columns + encoded_cat_names

print("Final feature count:", len(feature_order))

# Convert to Tensors

In [ ]:
X_train_tensor = torch.tensor(X_train_final, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train.values, dtype=torch.float32).view(-1, 1)

X_valid_tensor = torch.tensor(X_valid_final, dtype=torch.float32).to(device)
y_valid_tensor = torch.tensor(y_valid.values, dtype=torch.float32).view(-1, 1).to(device)

X_test_tensor = torch.tensor(X_test_final, dtype=torch.float32).to(device)
y_test_tensor = torch.tensor(y_test.values, dtype=torch.float32).view(-1, 1).to(device)

# Create Dataset & DataLoader

In [ ]:
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)

# Design Neural Network Architecture

In [ ]:
class InsuranceANN(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.ReLU(),
            nn.Dropout(0.2),

            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.2),

            nn.Linear(64, 32),
            nn.ReLU(),

            nn.Linear(32, 1),
        )

    def forward(self, x):
        return self.network(x)

In [ ]:
model = InsuranceANN(input_dim=X_train_final.shape[1]).to(device)

criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

model

# Train the Model

In [ ]:
epochs = 100
train_losses = []
valid_losses = []

for epoch in range(epochs):
    model.train()
    running_loss = 0.0

    for X_batch, y_batch in train_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)

        optimizer.zero_grad()
        preds = model(X_batch)
        loss = criterion(preds, y_batch)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * X_batch.size(0)

    epoch_train_loss = running_loss / len(train_dataset)
    train_losses.append(epoch_train_loss)

    model.eval()
    with torch.no_grad():
        valid_preds = model(X_valid_tensor)
        epoch_valid_loss = criterion(valid_preds, y_valid_tensor).item()
        valid_losses.append(epoch_valid_loss)

    if (epoch + 1) % 10 == 0:
        print(f"epoch {epoch + 1}/{epochs} - train loss: {epoch_train_loss:.2f} - valid loss: {epoch_valid_loss:.2f}")

In [ ]:
plt.figure(figsize=(9, 5))
plt.plot(train_losses, label="train loss")
plt.plot(valid_losses, label="validation loss")
plt.xlabel("epoch")
plt.ylabel("mse loss")
plt.title("Training Loss Curve")
plt.legend()
plt.tight_layout()
plt.savefig("../images/06_training_loss.png")
plt.show()

# Evaluate the Model

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

model.eval()
with torch.no_grad():
    test_preds = model(X_test_tensor).cpu().numpy().flatten()

y_test_actual = y_test.values

mae = mean_absolute_error(y_test_actual, test_preds)
rmse = np.sqrt(mean_squared_error(y_test_actual, test_preds))
r2 = r2_score(y_test_actual, test_preds)

print("Test MAE :", mae)
print("Test RMSE:", rmse)
print("Test R2  :", r2)

In [ ]:
plt.figure(figsize=(7, 7))
plt.scatter(y_test_actual, test_preds, alpha=0.4, color="#2563EB")
plt.plot(
    [y_test_actual.min(), y_test_actual.max()],
    [y_test_actual.min(), y_test_actual.max()],
    color="red",
    linestyle="--"
)
plt.xlabel("Actual Annual Medical Cost")
plt.ylabel("Predicted Annual Medical Cost")
plt.title("Actual vs Predicted")
plt.tight_layout()
plt.savefig("../images/07_actual_vs_predicted.png")
plt.show()

In [ ]:
residuals = y_test_actual - test_preds

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].scatter(test_preds, residuals, alpha=0.4, color="#16A34A")
axes[0].axhline(0, color="red", linestyle="--")
axes[0].set_xlabel("Predicted Value")
axes[0].set_ylabel("Residual")
axes[0].set_title("Residuals vs Predicted")

axes[1].hist(residuals, bins=50, color="#F97316")
axes[1].set_xlabel("Residual")
axes[1].set_title("Residual Distribution")

plt.tight_layout()
plt.savefig("../images/08_residual_analysis.png")
plt.show()

# Save Model & Artifacts

In [ ]:
import pickle

torch.save(model.state_dict(), "../models/medical_insurance_ann.pth")

with open("../artifacts/encoder.pkl", "wb") as f:
    pickle.dump(encoder, f)

with open("../artifacts/scaler.pkl", "wb") as f:
    pickle.dump(scaler, f)

feature_meta = {
    "numerical_columns": numerical_columns,
    "categorical_columns": categorical_columns,
    "feature_order": feature_order,
}

with open("../artifacts/feature_order.pkl", "wb") as f:
    pickle.dump(feature_meta, f)

print("model and artifacts saved")